<a href="https://colab.research.google.com/github/vedantx06/dreamlike-studio/blob/main/text_to_image_gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

try:
    # Attempt to get the actual GitHub token you used later in the notebook
    token = userdata.get('GITHUB_TOKEN')
    print('Successfully retrieved GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Error: The secret 'GITHUB_TOKEN' was not found.")
    print("To fix this, click the key icon (🔑) on the left sidebar and add a secret with the name 'GITHUB_TOKEN'.")

Successfully retrieved GITHUB_TOKEN


In [2]:
#diffusers is a hugging face page for using diffusion models from huggingface hub
!pip install diffusers transformers accelerate

In [3]:
!pip install matplotlib

In [4]:
from diffusers import StableDiffusionPipeline

import matplotlib.pyplot as plt
import torch

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [24]:
import torch
from diffusers import StableDiffusionPipeline

model_id1 = "dreamlike-art/dreamlike-diffusion-1.0"

# Automatically detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Initializing pipeline on {device}...")

# Initialize the StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    model_id1,
    torch_dtype=torch_dtype,
    use_safetensors=True
)

# Move to the detected device
pipe = pipe.to(device)

print(f"StableDiffusionPipeline initialized and moved to {device}.")

Initializing pipeline on cpu...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--dreamlike-art--dreamlike-diffusion-1.0/snapshots/9fb5a6463bf79d81152e715e8d2a8b988f96c790/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


StableDiffusionPipeline initialized and moved to cpu.


In [25]:
import torch
if torch.cuda.is_available():
    print(f"SUCCESS: GPU is active ({torch.cuda.get_device_name(0)})")
else:
    print("ERROR: GPU is not detected. Please go to Runtime > Change runtime type and select T4 GPU.")

ERROR: GPU is not detected. Please go to Runtime > Change runtime type and select T4 GPU.


In [7]:
!pip show torch

Name: torch
Version: 2.10.0+cpu
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision


In [12]:
import torch
if torch.cuda.is_available():
    print(f"SUCCESS: GPU is active ({torch.cuda.get_device_name(0)})")
else:
    print("ERROR: GPU is not detected. Please go to Runtime > Change runtime type and select T4 GPU.")

ERROR: GPU is not detected. Please go to Runtime > Change runtime type and select T4 GPU.


In [30]:
model_id1 = "dreamlike-art/dreamlike-diffusion-1.0"
model_id2 = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = StableDiffusionPipeline.from_pretrained(model_id1, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--dreamlike-art--dreamlike-diffusion-1.0/snapshots/9fb5a6463bf79d81152e715e8d2a8b988f96c790/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
prompt = """dreamlikeart, a grungy woman with rainbow hair, travelling between dimensions, dynamic pose, happy, soft eyes and narrow chin,
extreme bokeh, dainty figure, long hair straight down, torn kawaii shirt and baggy jeans
"""

In [ ]:
image = pipe(prompt).images[0]
image.save("dreamlike-diffusion-image.png")
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
prompt2 = """A girl is sittig on a chair & She is accompanied by her tiger. Make sure to keep it cinematic and color to be golden iris
"""

image = pipe(prompt2).images[0]

In [ ]:
print('[PROMPT]: ',prompt2)
plt.imshow(image);
plt.axis('off');

In [ ]:
prompt3 = """A girl is sittig on a chair & She is accompanied by her dog. Make sure to keep it cinematic and color to be golden iris
"""

image = pipe(prompt3).images[0]

In [ ]:
print('[PROMPT]: ',prompt3)
plt.imshow(image);
plt.axis('off');

In [ ]:
prompt4 = """A old beautiful lady is walking with her white little dog. Make sure to keep it cinematic and color to be golden iris
"""

image = pipe(prompt4).images[0]

In [ ]:
print('[PROMPT]: ',prompt4)
plt.imshow(image);
plt.axis('off');

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# Re-initialize the pipeline
model_id1 = "dreamlike-art/dreamlike-diffusion-1.0"
pipe = StableDiffusionPipeline.from_pretrained(model_id1, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")

print("Stable Diffusion Pipeline re-initialized on CUDA.")

In [21]:
import os
from flask import Flask, render_template, request, send_from_directory
import threading
import traceback
import torch
import socket

# Ensure clean directories
!rm -rf templates static
os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)

# Updated HTML with Loading State
index_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Dreamlike Studio Pro</title>
    <style>
        body { font-family: 'Segoe UI', sans-serif; background: #121212; color: white; text-align: center; padding-top: 50px; }
        .box { background: #1e1e1e; padding: 30px; border-radius: 15px; display: inline-block; width: 60%; box-shadow: 0 4px 15px rgba(0,0,0,0.5); }
        textarea { width: 90%; height: 80px; margin-bottom: 20px; border-radius: 5px; padding: 10px; background: #2c2c2c; color: white; border: 1px solid #333; }
        button { background: #03dac6; color: black; border: none; padding: 15px 30px; border-radius: 5px; cursor: pointer; font-weight: bold; margin: 5px; transition: 0.3s; }
        button:hover { opacity: 0.8; }
        .download-btn { background: #bb86fc; text-decoration: none; display: inline-block; }
        img { margin-top: 30px; max-width: 512px; border-radius: 10px; border: 2px solid #03dac6; }
        #loading { display: none; margin-top: 20px; color: #03dac6; font-weight: bold; }
    </style>
    <script>
        function showLoading() {
            document.getElementById('loading').style.display = 'block';
            document.getElementById('gen-form').style.opacity = '0.5';
        }
    </script>
</head>
<body>
    <div class='box'>
        <h1>Dreamlike Studio Pro</h1>
        <form id='gen-form' action='/generate' method='post' onsubmit='showLoading()'>
            <textarea name='prompt' placeholder='Describe your imagination...' required></textarea><br>
            <button type='submit'>Generate Image</button>
        </form>
        <div id='loading'>✨ Brewing your art... please wait (CPU is slow, may take 2-5 mins) ✨</div>
        {% if image_path %}
            <div><img src='/{{ image_path }}'></div>
            <div style='margin-top:20px;'>
                <a href='/{{ image_path }}' download='dreamlike_art.png'>
                    <button class='download-btn'>Download Image</button>
                </a>
            </div>
        {% endif %}
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(index_html)

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/static/<path:filename>')
def serve_static(filename):
    return send_from_directory('static', filename)

@app.route('/generate', methods=['POST'])
def generate():
    try:
        prompt_text = request.form.get('prompt')
        with torch.inference_mode():
            if device == 'cuda':
                with torch.autocast("cuda"):
                    img = pipe(prompt_text, num_inference_steps=20).images[0]
            else:
                img = pipe(prompt_text, num_inference_steps=20).images[0]

        img_path = "static/studio_out.png"
        img.save(img_path)
        return render_template('index.html', image_path=img_path)
    except Exception as e:
        traceback.print_exc()
        return f"Error: {str(e)}", 500

def get_free_port(start_port):
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                return port
            port += 1

free_port = get_free_port(5007)

def run_server(port):
    app.run(port=port, host='0.0.0.0', use_reloader=False)

threading.Thread(target=run_server, args=(free_port,), daemon=True).start()
print(f"Upgraded Flask server started on port {free_port}.")

 * Serving Flask app '__main__'
Upgraded Flask server started on port 5008.
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5008
 * Running on http://172.28.0.12:5008


In [22]:
from pyngrok import ngrok
from google.colab import userdata

# Restart Tunnel
ngrok.kill()
try:
    token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(token)
except:
    print('No NGROK_TOKEN found in Secrets. Using free tier.')

# Use the dynamic free_port from the Flask cell
try:
    current_port = free_port
except NameError:
    current_port = 5007

public_url = ngrok.connect(current_port)
print(f"Project is LIVE at: {public_url}")

Project is LIVE at: NgrokTunnel: "https://enable-deception-unsworn.ngrok-free.dev" -> "http://localhost:5008"


# Step 1: Initialize Git and Push to GitHub
This cell configures your identity and pushes the current code and static files to your repository.

In [ ]:
import os
from google.colab import userdata

# Configuration
GITHUB_USER = "vedantx06"
GITHUB_REPO = "dreamlike-studio"
token = userdata.get('GITHUB_TOKEN')

if not token:
    print("Error: GITHUB_TOKEN not found in Secrets.")
else:
    # Configure Git Identity
    !git config --global user.email "you@example.com"
    !git config --global user.name "Vedant"

    # Initialize/Refresh repository
    if not os.path.exists('.git'):
        !git init

    !git add .
    !git commit -m "Update: Added download functionality and web templates"

    # Set the remote URL with the token for authentication
    remote_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

    # Push to GitHub
    !git remote remove origin 2>/dev/null || true
    !git remote add origin {remote_url}
    !git branch -M main
    !git push -u origin main